# Visualize OT alignment before and after training

Notebook này so sánh **cùng một mẫu dịch, tokenizer, span source/target và cấu hình OT** giữa mô hình base và adapter/checkpoint sau Stage 1. Source và target luôn được lấy từ **cùng một forward pass**; không chạy hai câu riêng biệt.

Bốn heatmap chính gồm cosine similarity và transport plan trước/sau căn chỉnh. Ma trận chênh lệch cùng bảng chỉ số ở cuối hỗ trợ đọc kết quả định lượng.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'src').exists(), 'Hãy mở notebook từ thư mục repo hoặc notebooks/.'

# ===== Checkpoints =====
BEFORE_MODEL_NAME_OR_PATH = 'Qwen/Qwen2.5-0.5B'
AFTER_MODEL_OR_ADAPTER = str(REPO_ROOT / 'outputs' / 'stage1-multilingual-alignment')
TRUST_REMOTE_CODE = False

# ===== Một mẫu song ngữ để quan sát =====
SOURCE_LANG, TARGET_LANG = 'en', 'vi'
SOURCE_TEXT = 'The World Health Organization reported 42 attacks on health facilities in May 2019.'
TARGET_TEXT = 'Tổ chức Y tế Thế giới báo cáo 42 vụ tấn công vào các cơ sở y tế vào tháng 5 năm 2019.'
PROMPT_FORMAT = 'plain'       # 'plain' hoặc 'chat'
ENABLE_THINKING = False       # hữu ích với model như Qwen3
TRAINING_MODE = 'finetune'
ALIGNMENT_FORWARD_MODE = 'joint'  # 'joint' hoặc 'independent'
INDEPENDENT_ADD_SPECIAL_TOKENS = True

# ===== Phải khớp cấu hình Stage 1 khi muốn tái hiện chính xác OT loss =====
ALIGN_LAYER = -1
ALIGNMENT_FORWARD_MODE = 'joint'  # 'joint' hoặc 'independent'
INDEPENDENT_ADD_SPECIAL_TOKENS = True  # special tokens sẽ bị loại trước khi align
OT_SOLVER = 'sinkhorn'        # 'sinkhorn' hoặc 'ipot'
ATTENTION_MASS_WEIGHT = 0.5  # 0: uniform; 1: hoàn toàn theo attention
SINKHORN_EPSILON, SINKHORN_ITERATIONS = 0.1, 20
IPOT_BETA, IPOT_ITERATIONS, IPOT_INNER_ITERATIONS = 0.5, 50, 1

SAVE_FIGURES = False
FIGURE_DIR = REPO_ROOT / 'outputs' / 'ot-visualization'

In [ ]:
import gc
import sys
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from datasets import Dataset, DatasetDict
from peft import PeftConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

sys.path.insert(0, str(REPO_ROOT))
from src.prepare_data import prepare_alignment_dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = (torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
         else torch.float16 if torch.cuda.is_available() else torch.float32)
sns.set_theme(style='white')
print(f'Device: {DEVICE} | dtype: {DTYPE}')

## 1. Chuẩn bị đúng prompt và span như lúc train

Nếu `AFTER_MODEL_OR_ADAPTER` là PEFT adapter, notebook tự đọc base model từ `adapter_config.json`. Cả hai lượt đo dùng chung tokenizer để token và vị trí trong heatmap hoàn toàn tương ứng.

In [ ]:
after_path = Path(AFTER_MODEL_OR_ADAPTER)
is_adapter = after_path.is_dir() and (after_path / 'adapter_config.json').exists()
if is_adapter:
    adapter_config = PeftConfig.from_pretrained(AFTER_MODEL_OR_ADAPTER)
    resolved_base_model = adapter_config.base_model_name_or_path
    if BEFORE_MODEL_NAME_OR_PATH != resolved_base_model:
        print(f'Để so sánh công bằng, base được đổi thành model của adapter: {resolved_base_model}')
else:
    resolved_base_model = BEFORE_MODEL_NAME_OR_PATH

tokenizer_source = (AFTER_MODEL_OR_ADAPTER if after_path.is_dir() and
                    any((after_path / name).exists() for name in ['tokenizer.json', 'tokenizer_config.json'])
                    else resolved_base_model)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, trust_remote_code=TRUST_REMOTE_CODE)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

raw = DatasetDict({'test': Dataset.from_list([{
    'source': SOURCE_TEXT, 'target': TARGET_TEXT,
    'source_lang': SOURCE_LANG, 'target_lang': TARGET_LANG,
}])})
sample = prepare_alignment_dataset(
    raw, tokenizer, prompt_format=PROMPT_FORMAT,
    enable_thinking=ENABLE_THINKING, training_mode=TRAINING_MODE,
)['test'][0]

input_ids = torch.tensor(sample['input_ids'], dtype=torch.long).unsqueeze(0)
attention_mask = torch.ones_like(input_ids)
ss, se = sample['source_start_positions'], sample['source_end_positions']
ts, te = sample['target_start_positions'], sample['target_end_positions']
assert 0 <= ss < se <= ts < te <= input_ids.shape[1]

print('Full prompt:\n', tokenizer.decode(input_ids[0], skip_special_tokens=False))
print(f'\nSource span [{ss}:{se}]:', tokenizer.decode(input_ids[0, ss:se]))
print(f'Target span [{ts}:{te}]:', tokenizer.decode(input_ids[0, ts:te]))

In [ ]:
def readable_tokens(ids):
    tokens = tokenizer.convert_ids_to_tokens(ids.tolist())
    return [f'{i}: ' + t.replace('Ġ', '␠').replace('▁', '␠').replace('\n', '↵')
            for i, t in enumerate(tokens)]

source_tokens = readable_tokens(input_ids[0, ss:se])
target_tokens = readable_tokens(input_ids[0, ts:te])
pd.DataFrame({'source tokens': pd.Series(source_tokens), 'target tokens': pd.Series(target_tokens)})

## 2. Tái dựng phân phối khối lượng và transport plan

Code dưới đây giữ nguyên ý nghĩa của `src/model.py`, nhưng trả về cả ma trận vận chuyển để vẽ thay vì chỉ trả về scalar OT loss.

In [ ]:
def mixed_mass(scores, alpha):
    scores = scores.float().clamp_min(0)
    uniform = torch.full_like(scores, 1.0 / scores.numel())
    attention_mass = scores / scores.sum().clamp_min(1e-8)
    if scores.sum() <= 1e-8:
        attention_mass = uniform
    return alpha * attention_mass + (1.0 - alpha) * uniform

def sinkhorn_plan(cost, source_mass, target_mass, epsilon, iterations):
    log_a = source_mass.clamp_min(1e-38).log()
    log_b = target_mass.clamp_min(1e-38).log()
    log_k = -cost.float() / epsilon
    log_u = torch.zeros_like(source_mass, dtype=torch.float32)
    log_v = torch.zeros_like(target_mass, dtype=torch.float32)
    for _ in range(iterations):
        log_u = log_a - torch.logsumexp(log_k + log_v.unsqueeze(0), dim=1)
        log_v = log_b - torch.logsumexp(log_k.T + log_u.unsqueeze(0), dim=1)
    plan = torch.exp(log_u[:, None] + log_k + log_v[None, :])
    return plan / plan.sum().clamp_min(1e-8)

def ipot_plan(cost, source_mass, target_mass, beta, iterations, inner_iterations):
    log_a = source_mass.clamp_min(1e-38).log()
    log_b = target_mass.clamp_min(1e-38).log()
    log_kernel = -cost.float() / beta
    log_transport = log_a[:, None] + log_b[None, :]
    log_v = torch.zeros_like(target_mass, dtype=torch.float32)
    for _ in range(iterations):
        log_q = log_kernel + log_transport
        for _ in range(inner_iterations):
            log_u = log_a - torch.logsumexp(log_q + log_v.unsqueeze(0), dim=1)
            log_v = log_b - torch.logsumexp(log_q.T + log_u.unsqueeze(0), dim=1)
        log_transport = log_u[:, None] + log_q + log_v[None, :]
    plan = torch.exp(log_transport)
    return plan / plan.sum().clamp_min(1e-8)

def solve_plan(cost, source_mass, target_mass):
    if OT_SOLVER == 'ipot':
        return ipot_plan(cost, source_mass, target_mass, IPOT_BETA,
                         IPOT_ITERATIONS, IPOT_INNER_ITERATIONS)
    if OT_SOLVER == 'sinkhorn':
        return sinkhorn_plan(cost, source_mass, target_mass,
                             SINKHORN_EPSILON, SINKHORN_ITERATIONS)
    raise ValueError("OT_SOLVER phải là 'sinkhorn' hoặc 'ipot'")

## 3. Trích xuất trước và sau căn chỉnh

Hai model được nạp tuần tự để không giữ cả hai trong VRAM. `attn_implementation='eager'` là bắt buộc ở đây vì notebook cần attention tensor.

In [ ]:
def load_causal_lm(model_or_adapter, adapter=False, base_model_name=None):
    kwargs = dict(trust_remote_code=TRUST_REMOTE_CODE, attn_implementation='eager', torch_dtype=DTYPE)
    if DEVICE == 'cuda':
        kwargs['device_map'] = 'auto'
    if adapter:
        base = AutoModelForCausalLM.from_pretrained(base_model_name or resolved_base_model, **kwargs)
        model = PeftModel.from_pretrained(base, model_or_adapter)
    else:
        model = AutoModelForCausalLM.from_pretrained(model_or_adapter, **kwargs)
    if DEVICE == 'cpu':
        model = model.to(DEVICE)
    model.eval()
    return model

@torch.inference_mode()
def extract_alignment(model, forward_mode=None):
    forward_mode = ALIGNMENT_FORWARD_MODE if forward_mode is None else forward_mode
    if forward_mode not in {'joint', 'independent'}:
        raise ValueError("forward_mode must be 'joint' or 'independent'")
    input_device = model.get_input_embeddings().weight.device
    attention_layer = ALIGN_LAYER if ALIGN_LAYER < 0 else max(ALIGN_LAYER - 1, 0)

    def forward(ids, mask):
        return model(
            input_ids=ids.to(input_device), attention_mask=mask.to(input_device),
            output_hidden_states=True, output_attentions=True,
            use_cache=False, return_dict=True,
        )

    if forward_mode == 'joint':
        outputs = forward(input_ids, attention_mask)
        hidden = outputs.hidden_states[-1][0].float()
        src_hidden, tgt_hidden = hidden[ss:se], hidden[ts:te]
        src_ids = input_ids[0, ss:se].tolist()
        tgt_ids = input_ids[0, ts:te].tolist()
        attention = outputs.attentions[attention_layer][0].float().mean(dim=0)
        # Match src/model.py: attention received from target queries.
        received = attention[ts:te, :].sum(dim=0)
        source_scores, target_scores = received[ss:se], received[ts:te]
    else:
        src_batch = tokenizer(
            SOURCE_TEXT, add_special_tokens=INDEPENDENT_ADD_SPECIAL_TOKENS,
            return_tensors='pt', truncation=False,
        )
        tgt_batch = tokenizer(
            TARGET_TEXT, add_special_tokens=INDEPENDENT_ADD_SPECIAL_TOKENS,
            return_tensors='pt', truncation=False,
        )
        src_outputs = forward(src_batch['input_ids'], src_batch['attention_mask'])
        tgt_outputs = forward(tgt_batch['input_ids'], tgt_batch['attention_mask'])
        special_ids = set(tokenizer.all_special_ids)
        src_all_ids = src_batch['input_ids'][0].tolist()
        tgt_all_ids = tgt_batch['input_ids'][0].tolist()
        src_indices = [i for i, token_id in enumerate(src_all_ids) if token_id not in special_ids]
        tgt_indices = [i for i, token_id in enumerate(tgt_all_ids) if token_id not in special_ids]
        if not src_indices or not tgt_indices:
            raise ValueError('Source or target contains no non-special token')
        src_hidden_all = src_outputs.hidden_states[-1][0].float()
        tgt_hidden_all = tgt_outputs.hidden_states[-1][0].float()
        src_hidden_index = torch.tensor(src_indices, device=src_hidden_all.device)
        tgt_hidden_index = torch.tensor(tgt_indices, device=tgt_hidden_all.device)
        src_hidden = src_hidden_all.index_select(0, src_hidden_index)
        tgt_hidden = tgt_hidden_all.index_select(0, tgt_hidden_index)
        src_ids = [src_all_ids[i] for i in src_indices]
        tgt_ids = [tgt_all_ids[i] for i in tgt_indices]
        # There is no cross-attention between independent sequences. Each mass
        # therefore uses self-attention received inside its own sentence.
        src_attn = src_outputs.attentions[attention_layer][0].float().mean(dim=0)
        tgt_attn = tgt_outputs.attentions[attention_layer][0].float().mean(dim=0)
        src_attn_index = torch.tensor(src_indices, device=src_attn.device)
        tgt_attn_index = torch.tensor(tgt_indices, device=tgt_attn.device)
        source_scores = src_attn.index_select(0, src_attn_index).index_select(1, src_attn_index).sum(dim=0)
        target_scores = tgt_attn.index_select(0, tgt_attn_index).index_select(1, tgt_attn_index).sum(dim=0)

    similarity = F.normalize(src_hidden, dim=-1) @ F.normalize(tgt_hidden, dim=-1).T
    cost = 1.0 - similarity
    source_mass = mixed_mass(source_scores, ATTENTION_MASS_WEIGHT).to(cost.device)
    target_mass = mixed_mass(target_scores, ATTENTION_MASS_WEIGHT).to(cost.device)
    plan = solve_plan(cost, source_mass, target_mass)
    src_labels, tgt_labels = readable_tokens(torch.tensor(src_ids)), readable_tokens(torch.tensor(tgt_ids))

    probability = plan / plan.sum().clamp_min(1e-8)
    normalized_entropy = -(probability * probability.clamp_min(1e-38).log()).sum()
    normalized_entropy /= np.log(max(probability.numel(), 2))
    src_pos = torch.linspace(0, 1, len(src_ids), device=plan.device)[:, None]
    tgt_pos = torch.linspace(0, 1, len(tgt_ids), device=plan.device)[None, :]
    monotonic_band_mass = (plan * ((src_pos - tgt_pos).abs() <= 0.15)).sum()

    return {
        'alignment_forward_mode': forward_mode,
        'similarity': similarity.cpu().numpy(),
        'cost': cost.cpu().numpy(),
        'plan': plan.cpu().numpy(),
        'source_mass': source_mass.cpu().numpy(),
        'target_mass': target_mass.cpu().numpy(),
        'source_tokens': src_labels,
        'target_tokens': tgt_labels,
        'source_token_ids': src_ids,
        'target_token_ids': tgt_ids,
        'metrics': {
            'OT expected cost ↓': float((plan * cost).sum()),
            'Mean best cosine ↑': float(similarity.max(dim=1).values.mean()),
            'Normalized plan entropy': float(normalized_entropy),
            'Monotonic-band mass ↑ (heuristic)': float(monotonic_band_mass),
        },
    }

def run_and_release(model_path, adapter=False, base_model_name=None, forward_mode=None):
    model = load_causal_lm(model_path, adapter=adapter, base_model_name=base_model_name)
    result = extract_alignment(model, forward_mode=forward_mode)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

def analyze_checkpoint(model_path, adapter='auto', name=None, forward_mode=None):
    """Analyze one base model, full checkpoint, or PEFT adapter."""
    path = Path(str(model_path))
    if adapter == 'auto':
        adapter = path.is_dir() and (path / 'adapter_config.json').exists()
    base_model_name = None
    if adapter:
        base_model_name = PeftConfig.from_pretrained(str(model_path)).base_model_name_or_path
    print(f'Analyzing {name or model_path} | adapter={adapter}')
    result = run_and_release(str(model_path), adapter=bool(adapter),
                             base_model_name=base_model_name,
                             forward_mode=forward_mode)
    result['name'] = name or path.name or str(model_path)
    result['model_path'] = str(model_path)
    return result

def _word_groups(token_ids, tokenizer_obj):
    """Group common BPE/SentencePiece subwords while keeping CJK tokens readable."""
    token_strings = tokenizer_obj.convert_ids_to_tokens(token_ids)
    groups = []
    special_ids = set(tokenizer_obj.all_special_ids)
    for index, (token_id, token_string) in enumerate(zip(token_ids, token_strings)):
        piece = tokenizer_obj.decode([token_id], clean_up_tokenization_spaces=False)
        stripped = piece.strip()
        is_special = token_id in special_ids
        starts_word = token_string.startswith(('Ġ', '▁')) or (piece[:1].isspace())
        is_wordpiece_continuation = token_string.startswith('##')
        has_cjk = any(('\u3400' <= char <= '\u9fff') or
                      ('\uac00' <= char <= '\ud7af') for char in stripped)
        is_punctuation = bool(stripped) and all(
            unicodedata.category(char)[0] in {'P', 'S'} for char in stripped
        )
        start_new = (not groups or is_special or starts_word or is_punctuation or
                     (has_cjk and not is_wordpiece_continuation))
        if start_new:
            groups.append([index])
        else:
            groups[-1].append(index)
    labels = []
    for group in groups:
        ids = [token_ids[index] for index in group]
        label = tokenizer_obj.decode(ids, clean_up_tokenization_spaces=False).strip()
        labels.append(label or ''.join(token_strings[index] for index in group))
    return groups, labels

def merge_subword_transport_plan(result, tokenizer_obj=None):
    """Sum OT mass across subwords; total transport mass stays unchanged."""
    tokenizer_obj = tokenizer if tokenizer_obj is None else tokenizer_obj
    src_groups, src_labels = _word_groups(result['source_token_ids'], tokenizer_obj)
    tgt_groups, tgt_labels = _word_groups(result['target_token_ids'], tokenizer_obj)
    original = result['plan']
    merged = np.zeros((len(src_groups), len(tgt_groups)), dtype=original.dtype)
    for row, src_group in enumerate(src_groups):
        for column, tgt_group in enumerate(tgt_groups):
            merged[row, column] = original[np.ix_(src_group, tgt_group)].sum()
    assert np.isclose(merged.sum(), original.sum(), rtol=1e-5, atol=1e-7)
    return merged, src_labels, tgt_labels

def plot_transport_plan(result, title=None, ax=None, vmax=None, annotate=False,
                        cmap='magma', save_path=None, merge_subwords=True,
                        tokenizer_obj=None):
    """Plot one model's OT plan without BEFORE/AFTER dependencies."""
    if merge_subwords:
        plan, src_labels, tgt_labels = merge_subword_transport_plan(
            result, tokenizer_obj=tokenizer_obj
        )
    else:
        plan = result['plan']
        src_labels, tgt_labels = result['source_tokens'], result['target_tokens']
    owns_figure = ax is None
    if owns_figure:
        size = (min(max(10, len(tgt_labels) * 0.55), 30),
                min(max(6, len(src_labels) * 0.45), 24))
        fig, ax = plt.subplots(figsize=size)
    else:
        fig = ax.figure
    sns.heatmap(plan, ax=ax, cmap=cmap, vmin=0,
                vmax=float(plan.max()) if vmax is None else vmax,
                xticklabels=tgt_labels, yticklabels=src_labels, annot=annotate,
                fmt='.3f', cbar_kws={'label': 'transport mass'})
    model_name = result.get('name', result.get('model_path', 'model'))
    mode = result.get('alignment_forward_mode', 'unknown')
    ax.set_title(title or f'{model_name} - {OT_SOLVER.upper()} plan [{mode}]')
    ax.set(xlabel='Target tokens', ylabel='Source tokens')
    ax.tick_params(axis='x', rotation=70, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)
    if owns_figure:
        fig.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=180, bbox_inches='tight')
    if owns_figure:
        plt.show()
    return fig, ax

In [ ]:
print('Đang đo BEFORE:', resolved_base_model)
before = analyze_checkpoint(resolved_base_model, adapter=False, name='BEFORE')

print('Đang đo AFTER:', AFTER_MODEL_OR_ADAPTER)
after = analyze_checkpoint(AFTER_MODEL_OR_ADAPTER, adapter='auto', name='AFTER')
print('Hoàn tất trích xuất.')

## 4. Heatmap trước/sau

Hai similarity plot dùng chung thang màu; hai transport plot cũng dùng chung thang màu. Nhờ vậy độ đậm có thể so sánh trực tiếp giữa BEFORE và AFTER.

In [ ]:
def draw_heatmap(ax, matrix, title, cmap, vmin=None, vmax=None, center=None, cbar_label=None):
    sns.heatmap(matrix, ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, center=center,
                xticklabels=target_tokens, yticklabels=source_tokens,
                cbar_kws={'label': cbar_label} if cbar_label else None)
    ax.set_title(title)
    ax.set_xlabel('Target tokens')
    ax.set_ylabel('Source tokens')
    ax.tick_params(axis='x', rotation=70, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)

sim_min = min(before['similarity'].min(), after['similarity'].min())
sim_max = max(before['similarity'].max(), after['similarity'].max())
before_merged, before_src_words, before_tgt_words = merge_subword_transport_plan(before)
after_merged, after_src_words, after_tgt_words = merge_subword_transport_plan(after)
plan_max = max(before_merged.max(), after_merged.max())
width = min(max(14, len(before_tgt_words) * 0.70), 30)
height = min(max(10, len(before_src_words) * 0.55), 28)
fig, axes = plt.subplots(2, 2, figsize=(width, height), constrained_layout=True)
draw_heatmap(axes[0, 0], before['similarity'], 'BEFORE — cosine similarity',
             'viridis', sim_min, sim_max, cbar_label='cosine')
draw_heatmap(axes[0, 1], after['similarity'], 'AFTER — cosine similarity',
             'viridis', sim_min, sim_max, cbar_label='cosine')
plot_transport_plan(before, ax=axes[1, 0], vmax=plan_max)
plot_transport_plan(after, ax=axes[1, 1], vmax=plan_max)
fig.suptitle(f'OT token alignment: {SOURCE_LANG} → {TARGET_LANG}', fontsize=16)
if SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / 'ot_before_after.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
delta = after['similarity'] - before['similarity']
limit = max(abs(delta.min()), abs(delta.max()), 1e-6)
fig, ax = plt.subplots(figsize=(min(max(12, len(target_tokens) * 0.55), 30),
                                min(max(6, len(source_tokens) * 0.45), 20)))
draw_heatmap(ax, delta, 'Δ cosine similarity = AFTER − BEFORE',
             'coolwarm', -limit, limit, center=0, cbar_label='Δ cosine')
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(FIGURE_DIR / 'cosine_similarity_delta.png', dpi=180, bbox_inches='tight')
plt.show()

metrics = pd.DataFrame({'BEFORE': before['metrics'], 'AFTER': after['metrics']})
metrics['DELTA (after-before)'] = metrics['AFTER'] - metrics['BEFORE']
metrics

## Cách đọc kết quả

- `OT expected cost` giảm và `Mean best cosine` tăng là dấu hiệu trực tiếp nhất cho thấy representation của token nguồn/đích gần nhau hơn.
- Transport plan đậm, tập trung tại các cặp từ hợp lý cho thấy khối lượng được ghép rõ hơn. Entropy thấp chỉ có nghĩa plan tập trung hơn, **không tự động đồng nghĩa tốt hơn**.
- `Monotonic-band mass` chỉ là heuristic. Dịch có đảo trật tự từ hợp lệ, nên không nên ép mọi khối lượng nằm trên đường chéo.
- Tokenizer chia subword khác độ dài và các từ chức năng có thể tạo liên kết many-to-one/one-to-many. Vì vậy hãy kiểm tra nhiều mẫu và nhiều layer, không kết luận từ một heatmap.
- Chỉ so sánh hai lần chạy khi tokenizer, prompt, span, solver và tham số OT giống hệt nhau. Không so độ sắc của Sinkhorn với IPOT khi cấu hình khác nhau.